In [9]:
## solution to question 2.a
import random

def create_identity(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]

def show(M, label, p=3):
    print(f"\n{label}")
    for row in M:
        print("  [" + "  ".join(f"{x:8.{p}f}" for x in row) + "]")

def matrix_multiplication(A, B):
    rA, cA = len(A), len(A[0])
    cB = len(B[0])
    C = [[0.0] * cB for _ in range(rA)]
    for i in range(rA):
        for j in range(cB):
            s = 0.0
            for k in range(cA):
                s += A[i][k] * B[k][j]
            C[i][j] = s
    return C

def transpose(A):
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def max_err(A, B):
    return max(abs(A[i][j] - B[i][j])
               for i in range(len(A))
               for j in range(len(A[0])))


def make_spd(n):
    M = [[random.uniform(-3, 3) for _ in range(n)] for _ in range(n)]
    A = matrix_multiplication(transpose(M), M)
    for i in range(n):
        A[i][i] += n   # shift for clear positive definiteness
    return A

def elementary_matrix(n, i, j, factor):
    """Identity with −factor placed at (i, j).
       Represents the row op  R_i → R_i − factor·R_j."""
    E = create_identity(n)
    E[i][j] = -factor
    return E

def lu_decomposition(A):
    n = len(A)
    U = [row[:] for row in A]
    L = create_identity(n)
    E_matrices = []

    for j in range(n):                    # column j
        pivot = U[j][j]
        for i in range(j + 1, n):         # rows below the pivot
            factor = U[i][j] / pivot
            E = elementary_matrix(n, i, j, factor)
            E_matrices.append((i, j, factor, E))
            # apply E to U (in-place row update)
            for c in range(j, n):
                U[i][c] -= factor * U[j][c]
            # L collects the elimination factors
            L[i][j] = factor
    return L, U, E_matrices


##TEST
print("Q2a — LU on a 4x4 SPD matrix")
A = make_spd(4)
L, U, Es = lu_decomposition(A)
show(A, "A")
show(L, "L")
show(U, "U")
LU = matrix_multiplication(L, U)   # 1. compute the product
err = max_err(A, LU)               # 2. measure how close LU is to A
print("verified" if err < 1e-9 else "FAILED")

Q2a — LU on a 4x4 SPD matrix

A
  [  13.438     6.278    -0.824     2.992]
  [   6.278    13.547     6.719     7.249]
  [  -0.824     6.719    26.903    -5.615]
  [   2.992     7.249    -5.615    26.797]

L
  [   1.000     0.000     0.000     0.000]
  [   0.467     1.000     0.000     0.000]
  [  -0.061     0.669     1.000     0.000]
  [   0.223     0.551    -0.423     1.000]

U
  [  13.438     6.278    -0.824     2.992]
  [   0.000    10.614     7.104     5.851]
  [   0.000     0.000    22.099    -9.348]
  [   0.000     0.000     0.000    18.951]
verified


In [15]:
#Solution to 2.b Cholesky Decomposition
import math

def cholesky(A):
    n = len(A)
    L = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1):
            s = sum(L[i][k] * L[j][k] for k in range(j))
            if i == j:
                val = A[i][i] - s
                if val <= 0:
                    raise ValueError("not positive definite")
                L[i][j] = math.sqrt(val)
            else:
                L[i][j] = (A[i][j] - s) / L[j][j]
    return L

#Test
print("Q2b — Cholesky on the same A")
Lc = cholesky(A)
show(Lc, "L (Cholesky)")
LcT  = transpose(Lc)                          # 1. flip Lc across the diagonal
LLT = matrix_multiplication(Lc, LcT)          # 2. multiply Lc by Lᵀ
err = max_err(A, LLT)                        # 3. compare to A
print(f"max |A - L·Lᵀ| = {err:.2e}")
print("verified" if err < 1e-9 else "FAILED")



Q2b — Cholesky on the same A

L (Cholesky)
  [   3.666     0.000     0.000     0.000]
  [   1.713     3.258     0.000     0.000]
  [  -0.225     2.180     4.701     0.000]
  [   0.816     1.796    -1.988     4.353]
max |A - L·Lᵀ| = 3.55e-15
verified


In [17]:
##solution to 2.c


# --- Q2c, Q2d: QR ---
def qr_decomposition(A):
    m, n = len(A), len(A[0])
    a_cols = [[A[i][j] for i in range(m)] for j in range(n)]
    q_cols = []
    R = [[0.0] * n for _ in range(n)]
    for j in range(n):
        v = a_cols[j][:]
        for i in range(j):
            R[i][j] = sum(q_cols[i][k] * a_cols[j][k] for k in range(m))
            for k in range(m):
                v[k] -= R[i][j] * q_cols[i][k]
        norm = math.sqrt(sum(x * x for x in v))
        R[j][j] = norm
        q_cols.append([x / norm for x in v])
    Q = [[q_cols[j][i] for j in range(n)] for i in range(m)]
    return Q, R

##Test

print("\nQ2c — QR on a 4x2 example")
A2 = [[random.uniform(-5, 5) for _ in range(2)] for _ in range(4)]
Q, R = qr_decomposition(A2)
show(A2, "A"); show(Q, "Q"); show(R, "R")
print(f"max |A - QR| = {max_err(A2, matrix_multiplication(Q, R)):.2e}")



print("\nQ2d — QR on a random 7x5 matrix")
A75 = [[random.uniform(-5, 5) for _ in range(5)] for _ in range(7)]
Q, R = qr_decomposition(A75)
show(A75, "A (7x5)"); show(Q, "Q"); show(R, "R")
print("\nDiagonal of R:")
for i in range(5):
    print(f"  R[{i}][{i}] = {R[i][i]:.6f}")
print("\nObservation: all R[i][i] > 0  →  columns of A are linearly independent.")
print(f"max |A - QR| = {max_err(A75, matrix_multiplication(Q, R)):.2e}")



Q2c — QR on a 4x2 example

A
  [   2.070     4.082]
  [   4.467     2.193]
  [   4.994    -2.875]
  [   4.089    -2.714]

Q
  [   0.255     0.715]
  [   0.550     0.445]
  [   0.615    -0.387]
  [   0.504    -0.376]

R
  [   8.118    -0.888]
  [   0.000     6.026]
max |A - QR| = 0.00e+00

Q2d — QR on a random 7x5 matrix

A (7x5)
  [   2.573     2.092    -1.604    -0.063     3.904]
  [  -3.272     3.894    -4.742     3.241    -2.147]
  [   3.706     2.945     2.636    -1.294     2.585]
  [   1.954     3.989    -0.210    -0.245     0.433]
  [   4.745     3.090     3.550     1.492     0.016]
  [   1.865     1.116     4.624     3.789     0.629]
  [   0.703    -0.488     3.805    -2.098    -0.932]

Q
  [   0.328     0.142    -0.582     0.137     0.442]
  [  -0.418     0.823     0.135     0.092    -0.009]
  [   0.473     0.194     0.067    -0.380     0.394]
  [   0.249     0.476    -0.020    -0.321    -0.104]
  [   0.606     0.145     0.050     0.267    -0.679]
  [   0.238     0.042     0.6